In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve

from google.colab import files
uploaded = files.upload()

data = pd.read_csv("cleaned_merged_heart_dataset.csv")
data.head()

print(data.info())
print(data.describe())

# ------------------ Feature Description ------------------
print("\nHeart Disease Dataset Feature Description:\n")

feature_descriptions = {
    "age": "Age of patient",
    "sex": "Gender (1 = male, 0 = female)",
    "cp": "Chest pain type (0-3)",
    "trestbps": "Resting Blood Pressure",
    "chol": "Cholesterol level",
    "fbs": "Fasting blood sugar (1 = true, 0 = false)",
    "restecg": "ECG results (0-2)",
    "thalach": "Maximum heart rate achieved",
    "exang": "Exercise induced angina (1 = yes, 0 = no)",
    "oldpeak": "ST depression",
    "slope": "Slope of ST segment (0-2)",
    "ca": "Number of major vessels (0-3)",
    "thal": "Thalassemia (1-3)",
    "target": "Heart disease risk (1 = high, 0 = low)"
}

for feature, description in feature_descriptions.items():
    print(f"{feature.upper():<10} : {description}")

# ------------------ Data Visualization ------------------
plt.scatter(data["age"], data["target"], color="blue")
plt.title("Age vs Heart Disease")
plt.xlabel("Age")
plt.ylabel("Disease")
plt.show()

plt.scatter(data["chol"], data["target"], color="green")
plt.title("Cholesterol vs Heart Disease")
plt.xlabel("Cholesterol")
plt.ylabel("Disease")
plt.show()

plt.scatter(data["trestbps"], data["target"], color="red")
plt.title("BP vs Heart Disease")
plt.xlabel("BP")
plt.ylabel("Disease")
plt.show()

# ------------------ Prepare Data ------------------
X = data.drop("target", axis=1)
y = data["target"]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ------------------ Train Models ------------------
lr = LogisticRegression(max_iter=5000)
dt = DecisionTreeClassifier()
rf = RandomForestClassifier()
svm = SVC(probability=True)

lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
rf.fit(X_train, y_train)
svm.fit(X_train, y_train)

# ------------------ Accuracy ------------------
lr_acc = accuracy_score(y_test, lr.predict(X_test))
dt_acc = accuracy_score(y_test, dt.predict(X_test))
rf_acc = accuracy_score(y_test, rf.predict(X_test))
svm_acc = accuracy_score(y_test, svm.predict(X_test))

print("\nAccuracy Results:")
print("Logistic Regression:", lr_acc)
print("Decision Tree:", dt_acc)
print("Random Forest:", rf_acc)
print("SVM:", svm_acc)

# Training vs Testing Accuracy Graph
train_acc_lr = accuracy_score(y_train, lr.predict(X_train))
test_acc_lr = accuracy_score(y_test, lr.predict(X_test))

train_acc_dt = accuracy_score(y_train, dt.predict(X_train))
test_acc_dt = accuracy_score(y_test, dt.predict(X_test))

train_acc_rf = accuracy_score(y_train, rf.predict(X_train))
test_acc_rf = accuracy_score(y_test, rf.predict(X_test))

train_acc_svm = accuracy_score(y_train, svm.predict(X_train))
test_acc_svm = accuracy_score(y_test, svm.predict(X_test))

train_scores = [train_acc_lr, train_acc_dt, train_acc_rf, train_acc_svm]
test_scores = [test_acc_lr, test_acc_dt, test_acc_rf, test_acc_svm]

algorithms = ["Logistic", "Decision Tree", "Random Forest", "SVM"]

plt.plot(algorithms, train_scores, marker='o', label="Training Accuracy", color='blue')
plt.plot(algorithms, test_scores, marker='o', label="Testing Accuracy", color='red')

plt.title("Training vs Testing Accuracy")
plt.xlabel("Algorithms")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# ------------------ Cross Validation ------------------
cv_scores = cross_val_score(rf, scaler.fit_transform(X), y, cv=10)
print("\n10-Fold Cross Validation Accuracy:", cv_scores.mean())

# ------------------ Learning Curve ------------------
from sklearn.model_selection import learning_curve
import numpy as np

train_sizes, train_scores, test_scores = learning_curve(
    rf, X, y, cv=5, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10)
)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.plot(train_sizes, train_mean, label="Training Score", color='blue')
plt.plot(train_sizes, test_mean, label="Testing Score", color='red')

plt.title("Learning Curve - Random Forest")
plt.xlabel("Training Size")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# ------------------ Evaluation Metrics ------------------
y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("\n---- Evaluation Metrics (Random Forest) ----")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

# ------------------ Confusion Matrix ------------------
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    linewidths=0.5,
    linecolor='white'
)

plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ------------------ ROC Curve ------------------
y_probs = rf.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, y_probs)

plt.plot(fpr, tpr, color="purple")
plt.title("ROC Curve - Random Forest")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

# ------------------ Feature Importance ------------------
importance = rf.feature_importances_
features = X.columns

plt.barh(features, importance, color="purple")
plt.title("Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

# ------------------ Accuracy Bar Graph ------------------
plt.bar(
    ["Logistic", "Decision Tree", "Random Forest", "SVM"],
    [lr_acc, dt_acc, rf_acc, svm_acc],
    color=['royalblue', 'seagreen', 'darkorange', 'crimson']
)

plt.title("Accuracy Comparison of Algorithms")
plt.xlabel("Algorithms")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.show()

# ------------------ Pie Chart (Evaluation Metrics) ------------------
values = [accuracy, precision, recall, f1]
labels = ["Accuracy", "Precision", "Recall", "F1 Score"]

plt.figure(figsize=(6,6))

plt.pie(
    values,
    labels=labels,
    colors=['skyblue', 'lightgreen', 'yellow', 'lightcoral'],
    autopct='%1.1f%%',
    startangle=90
)

plt.title("Evaluation Metrics Pie Chart")
plt.axis('equal')
plt.show()

# ------------------ Prediction System ------------------
print("\nEnter New Patient Data")

age = int(input("Age: "))
sex = int(input("Sex (1=Male, 0=Female): "))
cp = int(input("Chest Pain Type (0-3): "))
trestbps = int(input("BP: "))
chol = int(input("Cholesterol: "))
fbs = int(input("Sugar (1/0): "))
restecg = int(input("ECG (0-2): "))
thalach = int(input("Heart Rate: "))
exang = int(input("Exercise Angina (1/0): "))
oldpeak = float(input("Oldpeak: "))
slope = int(input("Slope (0-2): "))
ca = int(input("Vessels (0-3): "))
thal = int(input("Thal (1-3): "))

new_data = pd.DataFrame([[age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal]],
                        columns=X.columns)

new_data = scaler.transform(new_data)

pred = rf.predict(new_data)

if pred[0] == 1:
    print("High Risk of Heart Disease")
else:
    print("Low Risk of Heart Disease")

# ------------------ Risk Pie Chart ------------------
risk_counts = data["target"].value_counts().sort_index()

labels = ["Low Risk", "High Risk"]
colors = ["lightgreen", "red"]

plt.figure(figsize=(6,6))

plt.pie(
    risk_counts,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90
)

plt.title("“Dataset Risk Distribution”")
plt.axis('equal')
plt.show()    on this  tooo  